[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r2/r2-q1/04_transfer_gap_measure.ipynb)

# R2-Q1 Week 3 — Measure the PV → PD transfer gap

This notebook applies the classifier you trained in Week 2 to PlantDoc,
measures the PV → PD accuracy gap using the statistical test you
pre-committed to in Week 1, and produces the headline finding for your
Week 3 draft presentation.

By the end of this notebook you will have:

- Per-image predictions on PV's test split (`pv_predictions.parquet`)
  and PD's test split (`pd_predictions.parquet`), with predicted class,
  ground-truth class, and the disease-category aggregation already
  joined in.
- A bootstrap distribution of the PV → PD gap (`transfer_gap.parquet`),
  one row per bootstrap iteration with PV-internal accuracy, PV → PD
  accuracy, and the gap.
- A reader-facing summary (`gap_summary.json`) with point estimates,
  95% confidence interval on the gap, and the verdict on whether the
  gap is statistically distinguishable from zero — the headline finding
  for your Week 3 draft presentation.

Week 4 picks up in Notebook 05 (`05_gap_characterization.ipynb`),
which uses `pd_predictions.parquet` to decompose the gap by host
group and by disease category.

## Before you run anything: switch to a GPU runtime

This notebook runs inference passes on the CNN classifier you trained
in Notebook 03 — once on PlantVillage's test split, once on PlantDoc's
test split — and then runs a bootstrap loop. Inference works on CPU
but takes 30–60 minutes per pass instead of the 3–5 minutes a T4 GPU
needs. The bootstrap is also faster on GPU. By default, Colab gives
you a CPU-only runtime.

You did this same runtime switch for Notebook 03 last week. If you're
opening this notebook in a fresh Colab session, you'll need to do it
again — Colab resets the runtime type between sessions.

To switch:

1. In Colab's menu bar: **Runtime → Change runtime type**.
2. Under "Hardware accelerator," choose **T4 GPU**.
3. Click **Save**.

Wait for the session to come up on the new runtime (a few seconds).
Then run the cells below in order.

**If Colab tells you "no GPUs available right now":** free-tier GPU
access is shared and occasionally unavailable. Wait a few minutes
and try again. If it keeps refusing, tell your mentor — the upgraded
Colab tiers have more reliable GPU access.

Setup Step 3 below requires a GPU runtime and will stop with a clear
error if one isn't attached. That error is your backstop in case you
skipped this markdown cell.

In [ ]:
try:
    import irilab2026 as iri
    print(f"OK — irilab2026 {iri.__version__} is installed and all deps importable.")
    print("    → use Pattern 2 in Cell 1:")
    print("      !pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main")
except ModuleNotFoundError as e:
    print(f"MISSING — {e.name} not importable.")
    print("    → use Pattern 1 in Cell 1:")
    print("      !pip install git+https://github.com/geraldmc/irilab2026.git@main")

In [ ]:
# Pattern 1 — fresh or partial runtime (installs deps that aren't present yet)
# This skips anything pip already sees as installed. This is ideal for a new environment or when you want
# to be sure everything is up to date.

# !pip install git+https://github.com/geraldmc/irilab2026.git@main

# Pattern 2 — populated runtime (refreshes irilab2026 only, leaves deps alone)
# This is ideal for when you already have the dependencies installed and just want to update irilab2026.

# !pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main

### Confirm the GPU is in place

Before going further, confirm Colab attached a GPU. The cell below
prints `GPU detected:` followed by `True` or `False`. If it's `True`,
you're good. If it's `False`, go back to the top of the notebook and
re-do the runtime switch — the next cells will not work without a GPU.

The check is a separate cell from the main setup (below) because it's
fast and recoverable: if the GPU isn't there, you see the problem
right away with no Drive authorization prompt to dismiss first.

In [ ]:
import irilab2026 as iri

gpu_ok = iri.has_gpu()
print(f"GPU detected: {gpu_ok}")

assert gpu_ok, (
    "No GPU detected. Go to the top of the notebook and follow the "
    "runtime-switch instructions, then re-run from Setup Step 1."
)

In [ ]:
# Mount Drive, set up irilab2026 with a GPU requirement, seed everything,
# and point OUTPUT_DIR at the R2-Q1 output directory.
iri.setup(gpu_required=True)
iri.seed_all(42)

OUTPUT_DIR = iri.output_dir("r2_q1")
print(f"Output directory: {OUTPUT_DIR}")

### Development mode

This notebook runs inference twice (on PV's test split, then on PD's
test split) and then runs a bootstrap loop with several thousand
iterations. End-to-end on full data takes 10–15 minutes on a Colab T4.

While you're iterating on the notebook — fixing typos, adjusting a
plot, debugging a save path — that round-trip cost adds up. The
`DEV_MODE` switch below gives you a faster development cycle by
shrinking everything proportionally:

- **`DEV_MODE = True`**: use PD's tiny variant (164 images), cap the
  bootstrap at 100 iterations regardless of your precommit, subsample
  the PV test set to 1,000 images. End-to-end run takes about 90
  seconds. The numbers you produce in dev mode are not paper-ready —
  they're just confirmation that the plumbing works.
- **`DEV_MODE = False`**: use PD's full test split, honor your
  precommit's `n_bootstrap`, run on the full PV test set. End-to-end
  run takes 10–15 minutes. These are the numbers you report.

Ship the notebook with `DEV_MODE = False` so a student opening it
fresh produces real numbers. Flip to `True` only while you're
iterating on something.

In [ ]:
### 1.1) DEV_MODE switch

DEV_MODE = False

if DEV_MODE:
    PD_VARIANT = "tiny"
    PV_SUBSAMPLE_N = 1_000   # number of PV test images to use in dev mode
    BOOTSTRAP_CAP = 100      # max bootstrap iterations regardless of precommit
    print("⚠️  DEV_MODE is ON — numbers produced are NOT paper-ready.")
    print(f"    PD variant: {PD_VARIANT}")
    print(f"    PV subsample: {PV_SUBSAMPLE_N:,} images")
    print(f"    Bootstrap cap: {BOOTSTRAP_CAP} iterations")
else:
    PD_VARIANT = "full"
    PV_SUBSAMPLE_N = None    # use full PV test split
    BOOTSTRAP_CAP = None     # honor precommit's n_bootstrap
    print("DEV_MODE is OFF — full production run.")
    print(f"    PD variant: {PD_VARIANT}")
    print(f"    PV subsample: full test split")
    print(f"    Bootstrap cap: honor precommit n_bootstrap")

## 2) Read your Week 1 pre-commits

In Week 1 you committed three methodology decisions to writing in
`precommit.json`: your aggregation level, your class-space remapping
(both PV and PD), and your statistical test choice with its
parameters. This notebook honors those commitments — it reads the file
at the top of the run and references the values from each section
where they apply.

If `precommit.json` doesn't exist in your R2-Q1 output directory, the
cell below fails with a message pointing you back to N02. You can't
run this notebook without your pre-commits.

In [ ]:
### 2.1) Read precommit.json
import json

precommit_path = OUTPUT_DIR / "precommit.json"

if not precommit_path.exists():
    raise FileNotFoundError(
        f"Couldn't find {precommit_path}. "
        "Run Notebook 02 (`02_pd_orientation.ipynb`) to completion — "
        "the last section writes precommit.json to this location."
    )

with open(precommit_path) as f:
    precommit = json.load(f)

# Validate the three top-level keys exist. Each downstream section
# cherry-picks the specific fields it needs from these.
expected_keys = {"aggregation_level", "class_space_remapping", "statistical_test"}
missing = expected_keys - set(precommit.keys())
if missing:
    raise KeyError(
        f"precommit.json is missing these top-level keys: {sorted(missing)}. "
        f"Re-run Notebook 02 to regenerate the file."
    )

print(f"Loaded: {precommit_path}")
print(f"  Top-level keys: {sorted(precommit.keys())}")

In [ ]:
### 2.2) Cherry-pick fields for downstream sections

# Section 4 (PV inference) and Section 5 (PD inference) use these mappings.
pv_class_to_category = precommit["class_space_remapping"]["pv_class_to_category"]
pd_class_to_category = precommit["class_space_remapping"]["pd_class_to_category"]

# Section 6 (bootstrap) uses these.
aggregation_choice = precommit["aggregation_level"]["choice"]
n_bootstrap = precommit["statistical_test"]["n_bootstrap"]
alpha = precommit["statistical_test"]["alpha"]
test_choice = precommit["statistical_test"]["choice"]

# Apply the DEV_MODE bootstrap cap if it's set.
if BOOTSTRAP_CAP is not None:
    if n_bootstrap > BOOTSTRAP_CAP:
        print(f"DEV_MODE: capping n_bootstrap from {n_bootstrap} to {BOOTSTRAP_CAP}")
        n_bootstrap = BOOTSTRAP_CAP

print("Pre-commits loaded:")
print(f"  Aggregation level:     {aggregation_choice}")
print(f"  PV → category map:     {len(pv_class_to_category)} entries")
print(f"  PD → category map:     {len(pd_class_to_category)} entries")
print(f"  Statistical test:      {test_choice}")
print(f"  Bootstrap iterations:  {n_bootstrap}")
print(f"  Significance level:    α = {alpha}")

Three notes on what these values commit you to and what they don't.

**`aggregation_choice` is informational here.** This notebook computes
accuracy at the disease-category level regardless of what you wrote —
the bootstrap operates on category-level matches, and the
characterization in Notebook 05 starts from category-level predictions.
The field is read here so it appears in `gap_summary.json` as
methodological provenance, not because the notebook branches on it.

**`test_choice` is also informational.** This notebook ships with a
two-sample bootstrap on the difference (PV-internal accuracy minus
PV → PD accuracy), regardless of what you committed to. That's the
single statistical test that R2-Q1's deliverables actually need.
If your precommit's `choice` field says something else, the notebook
records the discrepancy in `gap_summary.json` but doesn't branch
on it — the bootstrap on the difference is the most defensible
default and is the only path the notebook implements.

**The two mappings are load-bearing.** Sections 4 and 5 use
`pv_class_to_category` and `pd_class_to_category` to convert raw
class predictions and ground-truth labels into the category-level
match the bootstrap needs. If your mappings are missing entries —
say, you reassigned a PD class but forgot to do the same in PV —
Sections 4 and 5 will surface the missing-key error close to the
join. The validation here doesn't try to anticipate that.

## 3) Load the trained model

This section loads the CNN classifier you trained in Notebook 03. The
weights live in `baseline_resnet18.pt` in your R2-Q1 output directory —
Notebook 03's last section saved them there.

Three things happen below:

1. **Confirm the file exists.** If it doesn't, the cell raises a clear
   error pointing you back to N03.
2. **Rebuild the architecture and load the weights.** N03 saved only
   the model's state dict (the trained parameter values), not the
   architecture itself. We rebuild the same ResNet-18 with the same
   classifier head using `iri.build_baseline_model()`, then load the
   saved weights into it. The architecture has to match — if the
   weights file was trained on a different number of classes, the
   load fails with a size-mismatch error.
3. **Run a one-tensor smoke check.** A forward pass on a single
   dummy input confirms the model produces output of the expected
   shape and is actually running on the GPU. Cheap (under a second)
   and catches a "model loaded but doesn't behave right" failure
   here, rather than in the much slower Section 4 inference loop
   over thousands of real images.

`num_classes` for the rebuilt architecture comes from the length of
your PV class-to-category mapping in `precommit.json`. This is the
same number N03 trained against, since N02's mapping enumerates all
PV classes the model can produce.

In [ ]:
### 3.1) Load the trained baseline model

import torch

device = torch.device("cuda")

# Where the model lives — written by Notebook 03 at the end of training.
model_path = OUTPUT_DIR / "baseline_resnet18.pt"

if not model_path.exists():
    raise FileNotFoundError(
        f"Couldn't find {model_path}. "
        "Run Notebook 03 (`03_baseline_classifier.ipynb`) to completion — "
        "its final section saves the trained model to this location."
    )

# num_classes from the precommit's PV mapping. Same number N03 trained against.
num_classes = len(pv_class_to_category)

# Build the architecture, load the saved weights into it, switch to eval
# mode (disables dropout, freezes BatchNorm running stats), and move to GPU.
model = iri.build_baseline_model(num_classes=num_classes)
state_dict = torch.load(model_path, map_location=device, weights_only=True)
model.load_state_dict(state_dict)
model.eval()
model.to(device)

print(f"Loaded: {model_path}")
print(f"  file size:   {model_path.stat().st_size / 1e6:.1f} MB")
print(f"  num_classes: {num_classes}")
print(f"  device:      {device}")
print(f"  mode:        {'eval' if not model.training else 'train'}")  

In [ ]:
### 3.2) Smoke check — confirm output shape and device

# A single dummy input through the model. ResNet-18 expects 224x224 RGB.
# The point isn't to predict anything meaningful — it's to confirm the
# model is wired correctly and lives on the GPU.

with torch.no_grad():
    dummy_input = torch.zeros(1, 3, 224, 224, device=device)
    dummy_output = model(dummy_input)

assert dummy_output.shape == (1, num_classes), (
    f"Model output shape is {tuple(dummy_output.shape)}, "
    f"expected (1, {num_classes}). "
    "The saved weights may not match the architecture you just built — "
    "did you re-train N03 with a different number of classes?"
)
assert dummy_output.device.type == "cuda", (
    f"Model output is on {dummy_output.device}, expected cuda. "
    "model.to(device) may not have applied — re-run cell 3.1."
)

print("Smoke check passed:")
print(f"  output shape:  {tuple(dummy_output.shape)}")
print(f"  output device: {dummy_output.device}")
print(f"  output dtype:  {dummy_output.dtype}")

## 4) Run inference on PlantVillage's test split

This section produces `pv_predictions.parquet` — one row per PlantVillage
test image, with the model's prediction, the ground-truth label, the
softmax confidence, and the disease-category mapping from your precommit
applied to both prediction and truth.

The PV side of the transfer gap is the *baseline*: how well does the
model classify images from the same distribution it was trained on?
Section 6's bootstrap will compare this against PD accuracy (Section 5)
to compute the gap.

Three things happen below:

1. **Load PV's metadata, filter to the test split, optionally subsample
   for DEV_MODE.** Also builds a class_idx → class_label lookup and
   validates that PV's actual class set matches your precommit's
   PV mapping — catches the case where N02 was modified between runs.
2. **Run inference.** Standard PyTorch inference loop with the eval
   transform from the library, batch size 64, a progress bar so you
   can see it's moving.
3. **Assemble the predictions DataFrame and save.** The category
   mapping is applied here, the headline PV-internal accuracy is
   printed, and the file is round-trip verified before this section
   completes.

**Expected wall time:** with `DEV_MODE = True` (PV subsampled to
1,000 images), about 10–20 seconds. With `DEV_MODE = False`
(full 10,948 test images), about 2–5 minutes on a T4 — most of
the time is image decoding and the GPU transfer, not the forward
pass itself.

In [ ]:
### 4.1) Load PV test split and prepare lookups

# Load PV. If you ran Notebook 01 in this session, this is near-instant
# (cache hits); otherwise expect a 30-60 second download from HF Hub.
pv_metadata, pv_hf_dataset = iri.load_plantvillage()
pv_test = pv_metadata[pv_metadata["split"] == "test"].copy()

# Apply the DEV_MODE subsample if set. random_state=42 makes the subsample
# reproducible across runs.
if PV_SUBSAMPLE_N is not None:
    n_before = len(pv_test)
    pv_test = pv_test.sample(n=PV_SUBSAMPLE_N, random_state=42)
    print(f"DEV_MODE: subsampled PV test from {n_before:,} to {len(pv_test):,} rows")

# Build the idx -> label lookup. PV's class_idx is assigned by alphabetical
# sort over class_label, so this mapping is deterministic across runs.
pv_idx_to_label = dict(zip(pv_metadata["class_idx"], pv_metadata["class_label"]))

# Validate that PV's actual class set matches your precommit's PV mapping.
# Mismatch here means N02's pv_class_to_category drifted from PV's metadata —
# would silently break the category mapping in Cell 4.3 below if not caught.
metadata_labels = set(pv_metadata["class_label"].unique())
mapping_labels = set(pv_class_to_category.keys())
if metadata_labels != mapping_labels:
    only_in_metadata = sorted(metadata_labels - mapping_labels)
    only_in_mapping = sorted(mapping_labels - metadata_labels)
    raise ValueError(
        f"PV metadata and precommit mapping disagree on class labels.\n"
        f"  In PV metadata but not in your mapping: {only_in_metadata}\n"
        f"  In your mapping but not in PV metadata: {only_in_mapping}\n"
        "Re-run Notebook 02 — the PV mapping in your precommit doesn't "
        "match the dataset."
    )

print(f"PV test split:    {len(pv_test):,} rows")
print(f"PV classes:       {len(pv_idx_to_label)} (idx 0–{max(pv_idx_to_label)})")
print(f"Mapping aligned:  ✓")

In [ ]:
### 4.2) Run inference on PV test split

import numpy as np
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

BATCH_SIZE = 64
NUM_WORKERS = 2

# The eval transform: Resize(256) -> CenterCrop(224) -> ToTensor -> Normalize.
# Same pipeline N03 used during evaluation, lives in the library so it
# can't drift between training and inference.
eval_transform = iri.imagenet_eval_transform()
pv_dataset = iri.PlantVillageDataset(pv_test, pv_hf_dataset, transform=eval_transform)
pv_loader = DataLoader(
    pv_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

all_true_idx = []
all_pred_idx = []
all_confidence = []

with torch.no_grad():
    for images, labels in tqdm(pv_loader, desc="PV inference", leave=False):
        images = images.to(device, non_blocking=True)
        logits = model(images)
        probs = torch.softmax(logits, dim=1)
        pred_idx = logits.argmax(dim=1)
        confidence, _ = probs.max(dim=1)

        all_true_idx.append(labels.numpy())
        all_pred_idx.append(pred_idx.cpu().numpy())
        all_confidence.append(confidence.cpu().numpy())

true_idx = np.concatenate(all_true_idx)
pred_idx = np.concatenate(all_pred_idx)
confidence = np.concatenate(all_confidence)

assert len(true_idx) == len(pv_test), (
    f"Inference produced {len(true_idx)} predictions but PV test has "
    f"{len(pv_test)} rows. Something went wrong in the data loader."
)

print(f"Inference complete: {len(true_idx):,} predictions")

## 5) Run inference on PlantDoc's test split

This section produces `pd_predictions.parquet` — one row per PlantDoc
test image, with the model's prediction in PV's class space, PD's
ground-truth class label, the disease category from both perspectives,
and the host/disease/filename metadata Notebook 05 will need for its
cuts.

This is the *transfer side* of the gap. The model was trained on
PlantVillage; PlantDoc is a different distribution (field photos
rather than lab images). PV-internal accuracy from Section 4 told
you what's possible when the model is at home; PD accuracy from
this section tells you what happens when it isn't.

Three things happen below, parallel to Section 4 with three small
divergences:

1. **Load PD's metadata, filter to the test split.** DEV_MODE's
   `PD_VARIANT = "tiny"` switches to PD's 164-image stratified
   subsample (81 test images) — there's no row-count knob for PD
   the way there was for PV; tiny is the only fast path. Builds
   the validation check against `pd_class_to_category`, parallel
   to Section 4's check against `pv_class_to_category`.
2. **Run inference.** Same loop, same library transform, same batch
   size as Section 4. PD is small (236 test images full, 81 tiny)
   so this finishes in under 30 seconds regardless of mode.
3. **Assemble the predictions DataFrame.** Two mappings get applied:
   the *ground-truth* category comes from `pd_class_to_category`
   (PD's class space), and the *predicted* category comes from
   `pv_class_to_category` (the model predicts in PV's space). The
   precommit's dual-mapping schema is exactly what makes this work.
   Three PD-specific metadata columns (`host`, `disease`, `filename`)
   are joined in from PD's metadata so Notebook 05 doesn't have to
   re-load PD to do its cuts.

**Expected wall time:** 5–15 seconds in DEV_MODE (tiny variant, 81
test images), 15–30 seconds full (236 test images). Most of the
time is image decoding and download from HF Hub; the forward pass
itself is a fraction of a second.

In [ ]:
### 5.1) Load PD test split and prepare lookups

# Load PD. PD_VARIANT was set in Cell 8 by DEV_MODE — "tiny" in dev mode,
# "full" otherwise. First call downloads ~950 MB (full) or ~50 MB (tiny);
# subsequent calls in the session hit the cache.
pd_metadata, pd_hf_dataset = iri.load_plantdoc(variant=PD_VARIANT)
pd_test = pd_metadata[pd_metadata["split"] == "test"].copy()

# Validate that PD's actual class set matches your precommit's PD mapping.
# Same posture as Section 4.1's PV validation. We check against the full
# pd_metadata (not just pd_test) because the orphan class has 0 test images
# and would otherwise look like a "missing" entry.
metadata_labels = set(pd_metadata["class_label"].unique())
mapping_labels = set(pd_class_to_category.keys())
if metadata_labels != mapping_labels:
    only_in_metadata = sorted(metadata_labels - mapping_labels)
    only_in_mapping = sorted(mapping_labels - metadata_labels)
    raise ValueError(
        f"PD metadata and precommit mapping disagree on class labels.\n"
        f"  In PD metadata but not in your mapping: {only_in_metadata}\n"
        f"  In your mapping but not in PD metadata: {only_in_mapping}\n"
        "Re-run Notebook 02 — the PD mapping in your precommit doesn't "
        "match the dataset."
    )

print(f"PD variant:       {PD_VARIANT}")
print(f"PD test split:    {len(pd_test):,} rows")
print(f"PD classes:       {pd_metadata['class_label'].nunique()} (all variants)")
print(f"Classes in test:  {pd_test['class_label'].nunique()}")
print(f"Mapping aligned:  ✓")

## 6) Bootstrap the PV→PD gap

This is the section that produces the headline finding for your Week 3
draft presentation. The point estimate Section 5 printed
(PV-internal minus PV→PD) tells you *how big the gap looks*. The
bootstrap here tells you *whether you can claim the gap is real* — by
producing a 95% confidence interval on the gap and checking whether
that interval excludes zero.

The procedure follows the precommit you locked in N02: a two-sample
bootstrap on the difference. For each of `n_bootstrap` iterations:

1. Resample PV's predictions with replacement (same row count, same
   class distribution unchanged on average).
2. Resample PD's predictions with replacement, independently from PV.
3. Compute PV accuracy on the PV resample, PD accuracy on the PD
   resample, take the difference.

That gives you `n_bootstrap` plausible values for the gap, drawn from
the sampling distribution of the difference. The 95% CI is then
defined by the 2.5th and 97.5th percentiles of those values.

**Why two-sample on the difference rather than paired or two separate
CIs.** The PV and PD test sets aren't paired (different images,
different distributions), so paired bootstrap doesn't fit. Two separate
CIs would be more interpretable as a forest plot, but readers who don't
know that "overlapping CIs" doesn't imply "non-significant difference"
draw wrong conclusions; the CI on the gap itself avoids the trap.
Section 2 documents these choices for the paper.

**Expected wall time:** with `n_bootstrap = 1000`, about 10–30 seconds.
Most of the work is two numpy `random.choice` calls per iteration
plus two boolean-mean computations — all vectorizable, all fast.
With `DEV_MODE = True` and the bootstrap capped at 100 iterations,
the loop finishes in 1–3 seconds.

In [ ]:
### 6.1) Read predictions back from disk
import pandas as pd
# Re-reading from disk rather than reusing the in-memory DataFrames
# is deliberate — it confirms Section 4 and 5's parquet files are
# what they should be, and means the bootstrap doesn't depend on
# in-memory state from earlier in the run. If a student restarts
# the kernel between Section 5 and Section 6, this still works.
pv_predictions = pd.read_parquet(OUTPUT_DIR / "pv_predictions.parquet")
pd_predictions = pd.read_parquet(OUTPUT_DIR / "pd_predictions.parquet")

# The bootstrap only needs the boolean correctness vectors.
pv_correct = pv_predictions["correct_at_category"].values.astype(bool)
pd_correct = pd_predictions["correct_at_category"].values.astype(bool)

n_pv = len(pv_correct)
n_pd = len(pd_correct)

# Point estimates — the same numbers Section 4 and 5 printed, recomputed
# here to confirm reading the parquet didn't change anything.
pv_acc_point = pv_correct.mean()
pd_acc_point = pd_correct.mean()
gap_point = pv_acc_point - pd_acc_point

print(f"Read from disk:")
print(f"  PV predictions:  {n_pv:,} rows, accuracy {pv_acc_point:.4f}")
print(f"  PD predictions:  {n_pd:,} rows, accuracy {pd_acc_point:.4f}")
print(f"  Point gap:       {gap_point:+.4f}")
print()
print(f"Bootstrap config:")
print(f"  n_bootstrap:     {n_bootstrap:,}")
print(f"  alpha:           {alpha}")
print(f"  CI level:        {100 * (1 - alpha):.0f}%")

In [ ]:
### 6.2) Bootstrap n_bootstrap iterations

# Seeded RNG so the bootstrap is reproducible across runs. iri.seed_all(42)
# was called in Section 1; this RNG instance is for the bootstrap specifically
# so a re-run produces identical CI endpoints. If you ever need to verify
# robustness to seed choice, change the integer below and re-run.
rng = np.random.default_rng(seed=42)

pv_acc_boot = np.empty(n_bootstrap, dtype=np.float64)
pd_acc_boot = np.empty(n_bootstrap, dtype=np.float64)

# Pre-generate all the resample indices in two big arrays rather than
# regenerating each iteration's draw inside the loop. This is the standard
# numpy idiom for fast bootstrap — vectorize the index generation, loop
# only over the per-iteration accuracy computation.
pv_indices = rng.integers(0, n_pv, size=(n_bootstrap, n_pv))
pd_indices = rng.integers(0, n_pd, size=(n_bootstrap, n_pd))

for i in tqdm(range(n_bootstrap), desc="bootstrap", leave=False):
    pv_acc_boot[i] = pv_correct[pv_indices[i]].mean()
    pd_acc_boot[i] = pd_correct[pd_indices[i]].mean()

gap_boot = pv_acc_boot - pd_acc_boot

print(f"Bootstrap complete: {n_bootstrap:,} iterations")
print(f"  Gap distribution:")
print(f"    mean:     {gap_boot.mean():+.4f}")
print(f"    std:      {gap_boot.std():.4f}")
print(f"    min/max:  {gap_boot.min():+.4f} / {gap_boot.max():+.4f}")

In [ ]:
### 6.3) Compute confidence intervals, save, print verdict

# Percentile CIs. alpha=0.05 -> 2.5th and 97.5th percentiles.
pv_lo, pv_hi = np.percentile(pv_acc_boot, [100 * alpha/2, 100 * (1 - alpha/2)])
pd_lo, pd_hi = np.percentile(pd_acc_boot, [100 * alpha/2, 100 * (1 - alpha/2)])
gap_lo, gap_hi = np.percentile(gap_boot,   [100 * alpha/2, 100 * (1 - alpha/2)])

# Verdict: does the gap CI exclude zero? If both endpoints are on the
# same side of zero, the gap is statistically distinguishable from zero
# at the chosen alpha level.
ci_excludes_zero = (gap_lo > 0) or (gap_hi < 0)

# Save the bootstrap distribution. One row per iteration, three columns.
# Storing the full distribution lets you recompute any percentile, plot
# the histogram, or do downstream analyses without re-running the loop.
transfer_gap = pd.DataFrame({
    "pv_internal_accuracy": pv_acc_boot,
    "pv_to_pd_accuracy":    pd_acc_boot,
    "gap":                  gap_boot,
})

transfer_gap_path = OUTPUT_DIR / "transfer_gap.parquet"
transfer_gap.to_parquet(transfer_gap_path)

reloaded = pd.read_parquet(transfer_gap_path)
assert len(reloaded) == n_bootstrap, "Round-trip row-count mismatch"
assert list(reloaded.columns) == list(transfer_gap.columns), (
    "Round-trip column mismatch"
)

# Headline block.
print("=" * 60)
print("PV → PD TRANSFER GAP — HEADLINE RESULT")
print("=" * 60)
print()
print(f"PV-internal accuracy:  {pv_acc_point:.4f}  "
      f"[{pv_lo:.4f}, {pv_hi:.4f}]")
print(f"PV→PD accuracy:        {pd_acc_point:.4f}  "
      f"[{pd_lo:.4f}, {pd_hi:.4f}]")
print(f"Gap (PV − PV→PD):     {gap_point:+.4f}  "
      f"[{gap_lo:+.4f}, {gap_hi:+.4f}]")
print()
print(f"Bootstrap iterations:  {n_bootstrap:,}")
print(f"CI level:              {100 * (1 - alpha):.0f}%")
print(f"CI excludes zero:      {'YES — gap is significant' if ci_excludes_zero else 'NO — gap is not significant'}")
print()
print(f"Wrote: {transfer_gap_path}")
print(f"  size: {transfer_gap_path.stat().st_size / 1e3:.1f} KB")
print(f"  rows: {len(transfer_gap):,}")
print(f"  round-trip verified: ✓")

## 7) Synthesize `gap_summary.json`

This is the final analytical section. It pulls together everything
produced in Sections 4–6 — the point estimates, the CIs, the verdict
— and combines them with the methodological commitments you locked in
Week 1, plus a one-sentence interpretation suitable for quoting in
your Week 3 draft presentation.

The output is a single nested JSON file with three top-level sections:

- **`results`** — the numerical findings. PV-internal accuracy, PV→PD
  accuracy, the gap, all with point estimates and 95% confidence
  intervals; bootstrap distribution summary statistics; the verdict
  on whether the CI excludes zero.
- **`methodology`** — what you committed to in N02 (your aggregation
  level, the test choice, the alpha, the n_bootstrap), what you
  actually ran (the test that was implemented, which iterations
  count was used), and any flags about deviations from the precommit.
- **`interpretation`** — a single sentence stating the finding in
  prose, framed by magnitude rather than just the binary verdict.

You'll read this file when you write your Week 3 presentation: open
it in a viewer, copy the numbers from `results` into your slides,
copy the sentence from `interpretation` as your headline framing,
and use `methodology` to verify your presentation's methods slide
matches what you actually ran.

In [ ]:
### 7.1) Assemble the gap_summary dict

# The verdict sentence: magnitude-framed, paper-paragraph style.
# Built from variables already in memory so it can't drift from the
# numbers in `results`. Phrased in percentage points (not the raw
# proportion) because that's how the paper will quote it.
gap_pp = gap_point * 100
gap_lo_pp = gap_lo * 100
gap_hi_pp = gap_hi * 100

verdict_sentence = (
    f"A PlantVillage-trained ResNet-18 loses {gap_pp:.1f} percentage points "
    f"of accuracy when evaluated on PlantDoc rather than PlantVillage "
    f"(95% CI [{gap_lo_pp:.1f}, {gap_hi_pp:.1f}] from a "
    f"{n_bootstrap}-iteration bootstrap, both measured at the "
    f"disease-category level). The CI excludes zero, so the gap is "
    f"statistically distinguishable from no transfer loss."
)

# Track any deviations between what the precommit asked for and what
# the notebook actually ran. Two things to flag: the bootstrap cap
# (DEV_MODE only) and a test_choice mismatch.
deviations = []

precommit_n_bootstrap = precommit["statistical_test"]["n_bootstrap"]
if n_bootstrap != precommit_n_bootstrap:
    deviations.append(
        f"DEV_MODE: bootstrap iterations capped at {n_bootstrap} "
        f"(precommit specified {precommit_n_bootstrap})"
    )

if test_choice != "bootstrap_ci":
    deviations.append(
        f"Precommit test_choice was '{test_choice}', but this notebook "
        f"runs bootstrap_ci regardless. See Section 2 markdown for why."
    )

# Build the nested-by-domain dict.
gap_summary = {
    "results": {
        "pv_internal_accuracy": {
            "point": float(pv_acc_point),
            "ci_low": float(pv_lo),
            "ci_high": float(pv_hi),
            "n_observations": int(n_pv),
        },
        "pv_to_pd_accuracy": {
            "point": float(pd_acc_point),
            "ci_low": float(pd_lo),
            "ci_high": float(pd_hi),
            "n_observations": int(n_pd),
        },
        "gap": {
            "point": float(gap_point),
            "ci_low": float(gap_lo),
            "ci_high": float(gap_hi),
            "ci_excludes_zero": bool(ci_excludes_zero),
        },
        "bootstrap": {
            "n_iterations": int(n_bootstrap),
            "alpha": float(alpha),
            "ci_level": float(1 - alpha),
            "gap_mean": float(gap_boot.mean()),
            "gap_std": float(gap_boot.std()),
        },
    },
    "methodology": {
        "aggregation_level": {
            "committed": precommit["aggregation_level"]["choice"],
            "implemented": "disease_category",
        },
        "statistical_test": {
            "committed": precommit["statistical_test"]["choice"],
            "implemented": "bootstrap_ci",
            "alpha": float(alpha),
            "n_bootstrap_committed": int(precommit_n_bootstrap),
            "n_bootstrap_ran": int(n_bootstrap),
        },
        "class_space_remapping": {
            "n_pv_classes": len(pv_class_to_category),
            "n_pd_classes": len(pd_class_to_category),
            "categories": precommit["class_space_remapping"]["categories_used"],
        },
        "deviations_from_precommit": deviations,
    },
    "interpretation": verdict_sentence,
}

In [ ]:
### 7.2) Save gap_summary.json and render the headline

# Save with indent=2 so the file is human-readable when opened in a
# text editor or viewer. Trades a slight increase in file size for
# the ability to scan the structure without a JSON parser.
gap_summary_path = OUTPUT_DIR / "gap_summary.json"
with open(gap_summary_path, "w") as f:
    json.dump(gap_summary, f, indent=2)

# Round-trip verify.
with open(gap_summary_path) as f:
    reloaded = json.load(f)
assert reloaded == gap_summary, "Round-trip mismatch in gap_summary.json"

print(f"Wrote: {gap_summary_path}")
print(f"  size: {gap_summary_path.stat().st_size:,} bytes")
print(f"  top-level sections: {list(gap_summary.keys())}")
print(f"  deviations flagged: {len(deviations)}")
print(f"  round-trip verified: ✓")
print()

# Render the headline as it'll appear in the Week 3 presentation.
print("=" * 60)
print("WEEK 3 HEADLINE")
print("=" * 60)
print()
print(verdict_sentence)

## 8) What's next

Week 3 is done. Four files are now in your R2-Q1 output directory:

| File | Purpose |
|---|---|
| `pv_predictions.parquet` | Per-image predictions on PV's test split |
| `pd_predictions.parquet` | Per-image predictions on PD's test split, with host/disease/filename metadata |
| `transfer_gap.parquet` | The 1,000-iteration bootstrap distribution |
| `gap_summary.json` | The headline finding, paper-quotable verdict, methodology audit trail |

For your **Week 3 draft presentation**, the central artifact is
`gap_summary.json`. The `interpretation` field is your headline
sentence. The numbers in `results` go into your figures and tables.
The `methodology` section tells the audience what you committed to
and what you ran — your methods slide should match it.

For **Week 4**, you'll pick up in Notebook 05
(`05_gap_characterization.ipynb`), which uses
`pd_predictions.parquet` to decompose the gap by host group and by
disease category. The headline you have now is "a 34-point gap" — by
the end of N05 you'll be able to say *where* that gap is large and
*where* the model transfers cleanly.

One thing worth doing before you close this notebook: spot-check
`gap_summary.json` by opening it in a text editor (Drive's built-in
viewer works fine). The structure is small enough to read at a glance
and confirms that your Week 1 commitments propagated through correctly.
If anything in `methodology.deviations_from_precommit` is unexpected,
flag it with your mentor before quoting numbers in the presentation.